In [ ]:
# Importing needed code

from pathlib import Path
from typing import Callable, Literal
from datetime import datetime, time, date, timedelta
from multiprocessing.pool import Pool

import matplotlib as mpl
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import statistics as stat

from data_processing.processing.figure_of_merit import fit_fom, FOM, gaussian, n_sigma_classifier, bimodal
from data_processing.reporting.plotting import plot_fom, plot_scatter

from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.stats import linregress
from scipy.signal import savgol_filter

# Load Data

In [ ]:
# Data location

experiment_name = '2023-03-23_UBC_Background'
experiment_display_name = 'March 23 UBC Background'

In [ ]:
# Defining variables for important ARC folders

alloc_code = 'st-cberling-1'
project_folder = Path('/arc/project') / alloc_code
scratch_folder = Path('/scratch') / alloc_code
input_data_folder = project_folder / 'input_data'
output_data_folder = scratch_folder / 'output_data'
[x for x in scratch_folder.iterdir()]

In [ ]:
# Location of PSD data folder; change as needed
PARQ_ROOT = input_data_folder / experiment_name / 'processed_data/unfiltered/psd'

In [ ]:
# Location of experiment report folder
REPORT_ROOT = output_data_folder / 'analysis' / experiment_name
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
print(REPORT_ROOT.is_dir())

In [ ]:
STATS_REPORT = REPORT_ROOT / "stats_report.txt"

In [ ]:
%%time
# Load PSD data to "psd_report" DataFrame
psd_report = pd.read_parquet(PARQ_ROOT, columns=["CALIB_ENERGY","ENERGYSHORT", "ENERGY", "TIMETAG"])
psd_report = psd_report.astype({"CALIB_ENERGY": float, "ENERGYSHORT": int, "ENERGY": int, "TIMETAG": np.int64})

# Calculate PSD value as new column "tail / total"
psd_report["tail / total"] = (psd_report["ENERGY"] - psd_report["ENERGYSHORT"]) / psd_report["ENERGY"]
psd_report['TIMETAG_HOURS'] = psd_report['TIMETAG'] * 1e-12 / 3600
psd_report = psd_report.dropna()
psd_report = psd_report[psd_report["tail / total"].between(0,0.5)]

dataset_size = psd_report.shape[0]
print(f"Dataset size: {dataset_size:,d} rows")
psd_report.head()

In [ ]:
%%time
# Generate random sample of "psd_report" for graphing
# Graphing 300M points is way too slow, so best to graph a reasonable fraction of all points

sample_frac = 0.1  # fraction of points to put in sample (0-1)
random_state = 1323  # random number generator seed; change to try different samples

if sample_frac >= 1:
    graph_sample = psd_report
else:
    graph_sample = psd_report.sample(frac=sample_frac,random_state=random_state)

In [ ]:
# Define plotting variables

fontsize = 24
label_size = 22

In [ ]:
plot_width = 10
plot_height = 9
x_resolution = 256
count_limit = 5

y_resolution = x_resolution*plot_height//plot_width
max_energy = psd_report['CALIB_ENERGY'].max()

fig, ax = plt.subplots(figsize=(plot_width,plot_height))

cmap = mpl.colormaps['gnuplot']

ax.hist2d(
    psd_report['ENERGY'],
    psd_report['ENERGY'] - psd_report['ENERGYSHORT'],
    bins=(x_resolution, y_resolution),
    norm=mpl.colors.LogNorm(),
#     range=[[0, max_energy + .05], [0, 0.50]],
    cmap=cmap
)

# ax.set_ylim(0, 0.5)
# ax.set_xlim(0, max_energy + .05)
fig.suptitle("Tail vs Total", fontsize=fontsize+2)
ax.set_title(f"Event count = {dataset_size:,d}", fontsize=fontsize)
ax.set_xlabel("Tail", fontsize=fontsize)
ax.set_ylabel("Total", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)

In [ ]:
plot_width = 10
plot_height = 9
x_resolution = 256
count_limit = 5

y_resolution = x_resolution*plot_height//plot_width
max_energy = psd_report['CALIB_ENERGY'].max()

fig, ax = plt.subplots(figsize=(plot_width,plot_height))

cmap = mpl.colormaps['gnuplot']

ax.hist2d(
    psd_report['CALIB_ENERGY'],
    psd_report['tail / total'],
    bins=(x_resolution, y_resolution),
    norm=mpl.colors.LogNorm(),
    range=[[0, max_energy + .05], [0, 0.50]],
    cmap=cmap
)

ax.set_ylim(0, 0.5)
ax.set_xlim(0, max_energy + .05)
fig.suptitle("PSD vs Energy", fontsize=fontsize+2)
ax.set_title(f"Event count = {dataset_size:,d}", fontsize=fontsize)
ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
ax.set_ylabel("PSD", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)

## Heatmap & Histograms

In [ ]:
# Create 2D histogram of PSD vs Energy data
# Generates 2D grid of "bins

resolution = int(1024/2)

x, y = psd_report["CALIB_ENERGY"], psd_report["tail / total"]
Z, xe, ye = np.histogram2d(x, y, resolution)

In [ ]:
# Find energy slice width
energy_slice_width = xe[2] - xe[1]
f"Energy Slice Width = {energy_slice_width:.4f} MeVee"

In [ ]:
%%time
# Graph heatmap of 2D histogram
fig, ax = plt.subplots(figsize=(8,8))
cmap = plt.colormaps["nipy_spectral"]
pcm = ax.pcolormesh(xe, ye, Z.T, cmap=cmap)

fontsize = 24
ax.set_xlim(xe[0],1.5)
ax.set_title("Heatmap of Counts", fontsize=fontsize)
ax.set_ylabel("PSD", fontsize=fontsize)
ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)
fig.colorbar(pcm, ax=ax)

# FOM Analysis

In [ ]:
# TODO delete this when we get window using Californium source
# but keep in a FOM analysis notebook for when we need to recalibrate window

In [ ]:
# FOM function definition

BimodalParams = tuple[float, float, float, float, float, float]
GaussianParams = tuple[float, float, float]
BimodalBounds = tuple[BimodalParams, BimodalParams]

def split_params(
    params: BimodalParams
) -> tuple[GaussianParams, GaussianParams]:
    """Separates bimodal function parameters into 2 sets, one per component gaussian
    
    Parameters
    ----------
    params: BimodalParams
        parameters of a bimodal function
        
    Returns
    -------
    lower_gaussian_params: GaussianParams
        parameters of the lower gaussian (i.e. lower mu value)
    upper_gaussian_params: GaussianParams
        parameters of the upper gaussian (i.e. higher mu value)
    """
    params = abs(params)
    return params[0:3], params[3:]

def get_bimodal_fit(
    bins: np.ndarray, 
    histogram_slice: np.ndarray, 
    bounds: BimodalBounds
) -> tuple[GaussianParams, GaussianParams, np.ndarray]:
    """Fits a histogram slice to a bimodal distribution
    
    Parameters
    ----------
    bins: ndarray
        lower bounds of each PSD bin in the histogram
    histogram_slice: ndarray
        slice of the 2D PSD/Energy histogram taken for a specific energy (i.e. PSD vs Counts)
    bounds: BimodalBounds
        lower and upper bounds of fit parameters for this slice
        
    Returns
    -------
    gamma_params: GaussianParams
        parameters of the gaussian fit for gamma rays
    neutron_params: GaussianParams
        parameters of the gaussian fit for neutrons
    cov: ndarray
        estimated covariance of all bimodial parameters
    """
    params, cov = curve_fit(
        bimodal,
        bins,
        histogram_slice,
        bounds=bounds,
    )
    
    gamma_params, neutron_params = split_params(params)
    
    return gamma_params, neutron_params, cov


class SliceFitter:
    # based on work by Steven EngelHardt
    # https://www.stevenengelhardt.com/2013/01/16/python-multiprocessing-module-and-closures/
    def __init__(
        self, 
        bins: np.ndarray, 
        default_bounds: BimodalBounds,
        bounds: list[tuple[tuple[int, int], BimodalBounds]] | None = None
    ):
        self.bins = bins
        self.default_bounds = default_bounds
        self.bounds = bounds
        
    def __call__(self, numbered_slice):
        i, slice = numbered_slice
        fit_bounds = self.default_bounds

        if self.bounds is not None:
            for i_range, bound in self.bounds:
                if i in range(*i_range):
                    fit_bounds = bound

        gamma_params, neutron_params, cov = get_bimodal_fit(self.bins, slice, fit_bounds)

        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])

        perr = np.sqrt(np.diag(cov))

        return (i, *gamma_params, *neutron_params, fom), (i, *perr)


def scan_histogram_slices(
    bins: np.ndarray, 
    histogram: np.ndarray, 
    default_bounds: BimodalBounds,
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None = None, 
    start_idx: int = 0, 
    end_idx: int | None = None,
    cores: int = 4,
    use_chunks: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Determines bimodal fit and FOM for every energy slice in 2D PSD/Energy histogram
    
    Parameters
    ----------
    bins: ndarray
        lower bounds of each PSD bin in the histogram
    histogram: ndarray
        2D PSD/Energy histogram
    default_bounds: BimodalBounds
        default lower and upper bounds of fit parameters
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None, default None
        allows custom bounds for slice ranges. 
        Each list entry must have a tuple of start and stop indexes, and corresponding fit bounds.
        Bounds are used when the slice index falls within the start/stop range (start inclusive, stop exclusive).
        If index ranges overlap, the last matching range is used.
        If bounds is None, only default_bounds are used.
    start_idx: int, default 0
        starting index (inclusive) of slice range to fit to bimodal
    end_idx: int | None, default None
        ending index (exclusive) of slice range to fit to bimodal
        
    Returns
    -------
    fit_dataframe: DataFrame
        DataFrame of fit parameters including FOM (as columns) for each slice (as rows)
    error_dataframe: DataFrame
        DataFrame of (1 standard deviation) errors in fit parameters (as columns) for each slice (as rows)
    """
    end_idx = len(histogram) if end_idx is None else min(len(histogram), end_idx)
    pool_size = max(2*cores, 4)  # based on https://jupyter-tutorial.readthedocs.io/en/stable/performance/multiprocessing.html

    energy_slices = list(histogram[:, start_idx:end_idx].T)

    if use_chunks:
        chunksize, extra = divmod(len(energy_slices), pool_size*4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = 1
    
    pool = Pool(pool_size)
    results = pool.imap_unordered(
        SliceFitter(bins, default_bounds, bounds), 
        enumerate(energy_slices), 
        chunksize=chunksize)
    slice_params, slice_err = zip(*results)
    
    slice_params = sorted(list(slice_params), key=lambda x: x[0])
    slice_err = sorted(list(slice_err), key=lambda x: x[0])

    columns = ['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2']
    df = pd.DataFrame(slice_params, columns=columns + ['fom'])
    err_df = pd.DataFrame(slice_err, columns=columns)

    return df, err_df

In [ ]:
%%time
# Scan 2D histogram's energy slices and get bimodal fit

psd_bin_lbs = ye[:-1]

# Default
default_bounds = (
    (0.1, 0.01, 1, 
     0.25, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.38, 0.04, 2000)
)

bounds_a = (
    (0.1, 0.01, 1, 
     0.35, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.36, 0.04, 2000)
)

bounds_b = (
    (0.1, 0.01, 1, 
     0.34, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.36, 0.03, 2000)
)


# Ranged Example
bounds = [
    ((0,60), bounds_a),
]

start_scan_idx = 0
end_scan_idx = 420

end_scan_idx = min(end_scan_idx, len(Z))

# df_fom = find_threshold_fom_slice(psd_bin_lbs, Z.T, bounds, 0, end_scan_idx)

df, df_err = scan_histogram_slices(
    psd_bin_lbs, 
    Z.T,
    bounds=bounds,
    default_bounds=default_bounds, 
    start_idx = start_scan_idx, 
    end_idx = end_scan_idx
)


df.head()

In [ ]:
# Find energy slices with FOM between 1.2 and 1.3 (energy cutoff area of interest)

fom_slice_candidates = df.query("(1.2 < fom) & (fom <= 1.3)")
fom_slice_candidates

In [ ]:
# Manually find FOM threshold slice (number is index of first slice past threshold)

fom_index = 28
fom_slice = fom_slice_candidates.loc[fom_index,:]
slice_idx = int(fom_slice["i"])
fom = fom_slice["fom"]

In [ ]:
# Graph FOM threshold slice
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(psd_bin_lbs, Z.T[:,slice_idx], "k", lw=4)
params = tuple(df.iloc[slice_idx,1:7])
ax.plot(psd_bin_lbs, bimodal(psd_bin_lbs, *params), "r--", lw=4)
ax.set_ylim(1, 6e3)
ax.grid()
ax.set_yscale("log")
ax.set_ylabel("Counts", fontsize=fontsize)
ax.set_xlabel("PSD", fontsize=fontsize)
ax.set_title(f"E={xe[slice_idx]:.3f} MeVee; FOM={fom:.3f}", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=fontsize)
ax.tick_params(axis='both', which='minor', labelsize=fontsize)

In [ ]:
# Show histogram scan animation (optional, but useful for checking goodness of fit)
%matplotlib notebook
from matplotlib.animation import FuncAnimation

ylim = (1, 6e3)  # Y axis limits used (change to match data set used)
anim_fs = 20  # Font size used (axis labels and title)
anim_ls = 18  # Line size used for axis ticks
anim_lw = 3  # Line width used for line plotting
frames = range(0,120)  # slices to view
interval = 60  # delay between frames in milliseconds (lower = faster)

fig = plt.figure(figsize=(8,8))
ax = plt.axes(xlim=(0, 0.5), ylim=ylim, yscale="log")
line1, = ax.plot([], [], 'k', lw=anim_lw, label='Actual')
line2, = ax.plot([], [], 'r--', lw=anim_lw, label='Fit')

def init():
    line1.set_data([], [])
    line2.set_data([], [])
    ax.legend()
    ax.set_xlabel('PSD', fontsize=anim_fs)
    ax.set_ylabel('Counts', fontsize=anim_fs)
    ax.tick_params(axis='both', which='major', labelsize=anim_ls)
    ax.tick_params(axis='both', which='minor', labelsize=anim_ls)
    return line1, line2

def animate(i):
    x = psd_bin_lbs
    y = Z.T[:,i]
    line1.set_data(x, y)
    params = tuple(df.iloc[i,1:7])
    fom = df['fom'][i]
    y = bimodal(x, *params)
    line2.set_data(x, y)
    ax.set_title(f"E={xe[i]:.3f} MeVee; FOM={fom:.3f}", fontsize=anim_fs)

    return line1, line2

anim = FuncAnimation(fig, animate, init_func=init, frames=frames, interval=interval, blit=True)

# anim.save('../notebooks/images/FOM_graph.gif', writer='pillow')

In [ ]:
# Stops interactive graph views. Allows later static graphs, but stops animation
%matplotlib inline

In [ ]:
ylim = 1.2, 1.35  # Plot view limits on Y axis
xlim = 0.15, 0.25  # Plot view limits on X axis

fig, ax = plt.subplots(figsize=(8,8))


ax.hlines(1.27, 0, 1.25, color='k', linestyle='--', alpha=0.5, lw=4)

ax.plot(
    xe[start_scan_idx:end_scan_idx], 
    df['fom'][start_scan_idx:end_scan_idx], 
    "-",
    alpha=0.3,
    lw=4
)

ax.plot(
    xe[start_scan_idx:end_scan_idx], 
    df['fom'][start_scan_idx:end_scan_idx], 
    "ro",
    markersize=12,
)

ax.text(*(0.215, 1.27 + 0.01), "FOM=1.27", fontsize=fontsize)

ax.set_ylim(*ylim)
ax.set_xlim(*xlim)
ax.set_title("FOM vs Energy Slice", fontsize=fontsize)
ax.set_ylabel("FOM", fontsize=fontsize)
ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)

# ax.legend(fontsize=fontsize)
ax.grid()
# ax.set_yticks(np.arange(1,2.25,0.25))
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)

In [ ]:
# Interpolate between slices near FOM threshold to find best lower energy cutoff
fom_idx = slice_idx

fom_fit = interp1d(df["fom"][fom_idx-1:fom_idx+1], xe[fom_idx-1:fom_idx+1], kind="linear")
L0 = fom_fit(1.27)
f"FOM ENERGY CUTOFF = {L0:.4f} MeVee"

In [ ]:
# TODO get L0 from Californium source analysis
# for now, using value from "UBC Background 20221214-19 Update"
# TODO change this variable name to something more descriptive
L0 = 0.1966

# Counting Windows

In [ ]:
# Window fitting function definition

window_adj_offset = 0

def classify(
    psd_report: pd.DataFrame, 
    lb_fit_fn: Callable[[np.ndarray], np.ndarray], 
    ub_fit_fn: Callable[[np.ndarray], np.ndarray], 
    label: str, 
    le_cutoff: float = L0
) -> pd.DataFrame:
    """Classify signals as neutron or non-neutron for a given count window
    Classification creates a new column of boolean values, where True indicates a neutron classified signal.
    
    Parameters
    ----------
    psd_report: DataFrame
        DataFrame containing signal PSD data. 
        It must have the "tail / total" (for PSD value) and "CALIB_ENERGY" (for calibrated energy) columns.
    lb_fit_fn: Callable[[ndarray], ndarray]
        A function describing the window's lower bounds
    ub_fit_fn: Callable[[ndarray], ndarray]
        A function describing the window's upper bounds
    label: str
        column label to use for signal classification results
    le_cutoff: float, default L0 (previously determined lower energy cutoff)
        Lower energy cutoff (AKA the left boundary of the neutron window)
        
    Returns
    -------
    psd_report: DataFrame
        Original DataFrame with new column for neutron classification
    """
    psd_report[label] = psd_report["tail / total"].between(
        lb_fit_fn(psd_report["CALIB_ENERGY"].astype(float)) + 
        window_adj_offset, ub_fit_fn(psd_report["CALIB_ENERGY"].astype(float))
    ) & (psd_report["CALIB_ENERGY"] >= le_cutoff)
    
    return psd_report

def plot_classification(
    neutrons: pd.DataFrame, 
    gammas: pd.DataFrame, 
    lb_fit: Callable[[float], float],
    ub_fit: Callable[[float], float],
    max_energy: float,
    FOM_cutoff: float,
    n_neutrons: int,
    label_n: str = "Neutrons",
    label_g: str = "Non-Neutrons",
) -> tuple[plt.Figure, plt.Axes]:
    """Plot detected events, showing neutron classification window
    The classification window's upper and lower PSD bound functions, as well as the energy cutoff (left side) bound,
    are shown as dotted lines. Points are color labelled to show neutron vs. non-neutron events.
    
    Parameters
    ----------
    neutrons: pd.DataFrame
        DataFrame of neutron classified events to display
    gammas: pd.DataFrame
        DataFrame of non-neutron classified events to display (assumed to be gamma rays)
    lb_fit, ub_fit: Callabl[[float], float]
        Lower and upper PSD bound functions
    max_energy: float
        maximum energy used in this plot (which determines x-axis range)
    FOM_cutoff: float
        energy cutoff, i.e. lowest energy where FOM > 1.27
    n_neutrons: int
        Actual total number of neutrons (to be displayed in title)
        This parameter is used rather than obtained from the neutrons DataFrame
        to allow data samples to be plotted for faster plotting.
    label_n: str
        Series label for neutron classified events
    label_g: str
        Series label for non-neutron classified events (AKA gamma rays)
    
    Returns
    -------
    fig: plt.Figure
        plotting container for neutron classification plot
    ax: plt.Axes
        plot axes for neutron classification plot
    
    """
    fig, ax = plt.subplots(figsize=(10,10))
    ax.scatter(gammas["CALIB_ENERGY"], gammas["tail / total"], s=2, label=label_g)
    ax.scatter(neutrons["CALIB_ENERGY"], neutrons["tail / total"], s=2, label=label_n)

    energy_space = np.linspace(xe[0], max_energy + 0.5, 200)

    ax.plot(energy_space, lb_fit(energy_space), 'r--')
    ax.plot(energy_space, ub_fit(energy_space), 'r--')
    
    ax.vlines(all_slice_xs[0], lb_fit(all_slice_xs[0]), ub_fit(all_slice_xs[0]), 'r', ls='--')

    ax.vlines(FOM_cutoff, lb_fit(FOM_cutoff), ub_fit(FOM_cutoff), 'r', ls="--")

    ax.set_title(f"Counts = {n_neutrons}", fontsize=fontsize)
    ax.set_ylim(0, 0.55)
    ax.set_xlim(0, max_energy + .05)
    
    ax.tick_params(axis='both', which='major', labelsize=22)
    ax.tick_params(axis='both', which='minor', labelsize=22)
    
    ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.legend(fontsize=fontsize)
    return fig, ax

In [ ]:
# Define variable for all slice indexes to use
# (can be overridden if you only want to plot a subset of x axis)

all_slice_xs = xe[:end_scan_idx]

## NASA  


This classification window is based on the technique outlined in _Fast Neutron Spectroscopy With Organic Scintillation Detectors in a High-Radiation Environment_, Baramsai _et al_.

We define the lower PSD bound as N standard deviations above the gamma mean. We find these points for each slice, then use smoothing and interpolation to generate a lower bound function.

The upper bound for the window is generated by adding a fixed offset to the lower bound value (found to be 0.2 based on observation of the paper's graphs).

As in all windows, the FOM cutoff is used as the left-most bound.

In [ ]:
# Define lower and upper bounds (neutron_lb_fit, neutron_ub_fit)

window_offset = 0.2
sigma = 5

neutron_lb = savgol_filter(df["mu1"] + sigma * df["sigma1"], window_length=21, polyorder=3)  # reduce noise
neutron_lb_fit = interp1d(
    all_slice_xs, 
    neutron_lb, 
    fill_value=(neutron_lb[0], neutron_lb[-1]), 
    bounds_error=False)  # now it's a function!

def neutron_ub_fit(x):
    # upper bound is just a fixed PSD offset from lower bound
    # as observed in NASA paper graphs
    return neutron_lb_fit(x) + window_offset

In [ ]:
# Plot upper and lower bounds (diagnostic)

plt.plot(all_slice_xs, neutron_lb_fit(all_slice_xs))
plt.plot(all_slice_xs, neutron_ub_fit(all_slice_xs))

In [ ]:
%%time
# Classify neutrons under NASA window
psd_report = classify(psd_report, neutron_lb_fit, neutron_ub_fit, "NASA",le_cutoff=L0)

In [ ]:
%%time
# Recreate graph sample with classification data
del graph_sample
graph_sample = psd_report.sample(frac=sample_frac,random_state=random_state)

In [ ]:
plot_width = 10
plot_height = 9
x_resolution = 256
count_limit = 5

y_resolution = x_resolution*plot_height//plot_width
max_energy = psd_report['CALIB_ENERGY'].max()

fig, ax = plt.subplots(figsize=(plot_width,plot_height))
g_vs_n = psd_report['NASA'].map({True: 1, False: -1})

cmap = mpl.colormaps['RdBu_r']

ax.hist2d(
    psd_report['CALIB_ENERGY'],
    psd_report['tail / total'],
    weights=g_vs_n,
    bins=(x_resolution, y_resolution),
    range=[[0, max_energy + .05], [0, 0.50]],
    cmap=cmap,
    vmin=-count_limit,
    vmax=count_limit,
)

energy_space = np.linspace(0, max_energy + 0.5, 200)
ax.plot(energy_space, neutron_lb_fit(energy_space), 'r--')
ax.plot(energy_space, neutron_ub_fit(energy_space), 'r--')
ax.vlines(all_slice_xs[0], neutron_lb_fit(all_slice_xs[0]), neutron_ub_fit(all_slice_xs[0]), 'r', ls='--')
ax.vlines(L0, neutron_lb_fit(L0), neutron_ub_fit(L0), 'r', ls="--")

ax.set_ylim(0, 0.5)
ax.set_xlim(0, max_energy + .05)
n_neutrons = psd_report[psd_report["NASA"]].shape[0]
fig.suptitle(f"Neutron Classification: {experiment_display_name}", fontsize=fontsize+2)
ax.set_title(f"Neutron count = {n_neutrons}", fontsize=fontsize)
ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
ax.set_ylabel("PSD", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)
event_colors = [mpl.patches.Patch(facecolor=cmap(1.)),
                mpl.patches.Patch(facecolor=cmap(0.))]
ax.legend(event_colors, ["Neutrons", "Gamma"])

fig.savefig(REPORT_ROOT / 'classification.png')

# CPS Analysis

In [ ]:
%%time
# Defining count rate functions

# TODO redevelop to work with on/off time ranges
# any counts outside the ranges should be NaN
def counts_over_time_histogram(
    timetags: pd.Series, 
    dwell_time: int, 
    total_time: float
) -> tuple[np.ndarray, np.ndarray]:
    """Generates histogram of event counts per dwell time bin
    Due to limitations of data acquisition, the first bin is ignored, as it gives inaccurately low values.
    
    Parameters
    ----------
    timetags: pd.Series
        timetags for all relevant events 
    dwell_time: int
        time period used for histogram bin width in seconds
    total_time: float
        total data acquisition time in seconds
        
    Returns
    -------
    counts: np.ndarray
        array of counts per bin
    bins: np.ndarray
        bin boundaries (as time in seconds)
    """
    if total_time % dwell_time == 0:
        total_time += 0.001
    input_bins = np.arange(0, total_time, dwell_time)
    counts, bins = np.histogram(timetags, input_bins)
    return counts[1:], bins[1:]


def plot_histogram(
    bins: list[np.ndarray], 
    counts: list[np.ndarray],
    labels: list[str],
    style: Literal['points', 'stairs'] = "points",
    **kwargs
) -> tuple[plt.Figure, plt.Axes]:
    """Plot a set of histograms in one figure
    
    Parameters
    ----------
    bins: list[np.ndarray]
        list of histogram bins to use
    counts: list[np.ndarray]
        list of histogram counts to use
        Each list element must correspond to the element in the same position in `bins`
    labels: list[str]
        list of labels to use
        Each list element must correspond to the element in the same position in `bins`
    style: Literal['points', 'stairs'], default "points"
        plotting style
        The 'points' option creates a scatterplot, while 'stairs' creates a stair plot.
    kwargs
        Other formatting options used in the Matplotlib Axes.plot() or Axes.stairs() functions (as appropriate)
    
    Returns
    -------
    fig: plt.Figure
        plotting container for histogram plot
    ax: plt.Axes
        plot axes for histogram plot
    
    """
    fig, ax = plt.subplots(dpi=140)
    
    for bin, count, label in zip(bins, counts, labels):
        if style == "points":
            ax.plot(bin[:-1], count, "o", label=label, **kwargs)
        
        elif style == "stairs":
            ax.stairs(count, bin, label=label, lw=4, **kwargs)
    
    ax.legend()
    return fig, ax

In [ ]:
# Define time in hours for better plotting
# TODO move this earlier in notebook

psd_report['TIMETAG_HOURS'] = psd_report['TIMETAG'] * 1e-12/3600

In [ ]:
# Display total time
total_time = psd_report["TIMETAG"].max() * 1e-12 # s
f"Total time = {total_time:.3f} s = {total_time/60:.3f} min = {total_time/3600:.3f} hrs"

In [ ]:
psd_neutrons = psd_report[psd_report['NASA']]
psd_gammas = psd_report.query("not NASA")

### Overall Statistics

In [ ]:
# Find overall on/off counts
# TODO change variable name (total?)


count_neutrons = psd_neutrons.shape[0]
count_gamma = psd_gammas.shape[0]
total_counts = psd_report.shape[0]

report_lines = [
    f"Statistics Report: {experiment_display_name}",
    "",
    "Overall Count Statistics",
    "------------------------",
    f"Overall count (Neutrons): {count_neutrons}",
    f"Overall count (Gamma rays): {count_gamma}",
    f"Total count: {total_counts}",
    "------------------------",
    ""
]
report_str = "\n".join(report_lines)

with open(STATS_REPORT, "w") as reportfile:
    reportfile.write(report_str)
    
print(report_str)

### Count rates

In [ ]:
# Define dwell time
dwell_time = 60*1 # s

In [ ]:
%time
# Generate counts per dwell time

dwell_time_hours = dwell_time / 3600
total_time_hours = total_time / 3600

neutron_time_counts, neutron_time_bins = counts_over_time_histogram(
    psd_neutrons["TIMETAG_HOURS"], dwell_time_hours, total_time_hours
)

gamma_time_counts, gamma_time_bins = counts_over_time_histogram(
    psd_gammas["TIMETAG_HOURS"], dwell_time_hours, total_time_hours
)

In [ ]:
# Generate count rates for each dwell time bin
neutron_time_cps = neutron_time_counts / dwell_time
gamma_time_cps = gamma_time_counts / dwell_time

In [ ]:
# Determine count rate mean and standard deviation (2 methods)

neutron_mean = np.average(neutron_time_cps)
neutron_count_sigma = np.sqrt(count_neutrons)/total_time
neutron_cps_sigma = stat.stdev(neutron_time_cps)

report_lines = [
    "Count Rate Statistics",
    "------------------------",
    f"Mean Count Rate: {neutron_mean:.4f} 1/s",
    f"Standard Deviation (over all dwell rates): {neutron_cps_sigma:.4f} 1/s",
    f"Count Rate Sigma (using total count): {neutron_count_sigma:.4f} 1/s",
    "------------------------",
    ""
]
report_str = "\n".join(report_lines)

with open(STATS_REPORT, "a") as reportfile:
    reportfile.write(report_str)
    
print(report_str)

In [ ]:
%%time
# Plot neutron and gamma count rate over experiment run time
fig, ax = plot_histogram(
    [neutron_time_bins, gamma_time_bins],
    [neutron_time_cps, gamma_time_cps],
    labels=["Neutron CPS", "Non-Neutron CPS"],
    markersize =3,
)



# ax.set_yscale('log')
ax.grid()
fig.suptitle("Event Count Rates over Experiment Time")
ax.set_title(f"Dwell time = {dwell_time} s")
ax.set_ylabel("Count Rate, 1/s", fontsize=14)
ax.set_xlabel("Time (hrs)", fontsize=14)
# ax.set_ylim(0, 10)
#ax.set_xlim(0.075, 0.25)

In [ ]:
# Plot neutron count rate only over experiment run time
fig, ax = plot_histogram(
    [neutron_time_bins],
    [neutron_time_cps],
    labels=["Neutron CPS"],
    markersize=3,
)

#ax.set_yscale('log')
ax.grid()
fig.suptitle("Neutron Count Rate over Experiment Time")
ax.set_title(f"Dwell time = {dwell_time} s")
ax.set_ylabel("Count Rate, 1/s", fontsize=14)
ax.set_xlabel("Time (hrs)", fontsize=14)
#ax.set_xlim(0.075, 0.25)

fig.savefig(REPORT_ROOT / 'count_rate_over_time.png')

In [ ]:
# Create count rate histogram for neutrons and gamma rays
cps_n_bins = 20

neutron_cps_counts, neutron_cps_bins = np.histogram(filtered_n_cps, cps_n_bins)
gamma_cps_counts, gamma_cps_bins = np.histogram(filtered_g_cps, cps_n_bins)


f"CPS Bin Width: {gamma_cps_bins[2] - gamma_cps_bins[1]:.4f} 1/s"

In [ ]:
max(neutron_cps_counts)

In [ ]:
# TODO determine if this is needed (since similar code exists above)
# neutron_mean = np.average(neutron_time_cps)
# neutron_sigma = np.sqrt(neutron_mean / len(neutron_time_cps))
# print(f"Mean: {neutron_mean}\nSigma: {neutron_sigma}")
# print(neutron_time_counts)

In [ ]:
# Plot count rate histogram for neutrons
fig, ax = plot_histogram(
    [neutron_cps_bins, gamma_cps_bins],
    [neutron_cps_counts, gamma_cps_counts],
    style="stairs",
    labels=["Neutrons"]
)


ax.grid()
ax.set_xlabel("CPS")
ax.set_ylabel("Counts")
fig.suptitle("Neutron CPS histogram")
ax.set_title(f"Zero filtered, Dwell time = {dwell_time}s, CPS bins = {cps_n_bins}")
# ax.set_xlim(0.02, 0.45)
# ax.set_ylim(0,10)

fig.savefig(REPORT_ROOT / 'neutron_cps_histogram.png')